In [7]:
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
import numpy as np
import time


TILE_SIZE = 512
THRESHOLD = 0.15
B04_T0 = r"E:\Satellite data\S2C_MSIL2A_20250330T045721_N0511_R119_T44QPE_20250330T100109.SAFE\GRANULE\L2A_T44QPE_A002949_20250330T051221\IMG_DATA\R10m\B04_10m.jp2"
B08_T0 = r"E:\Satellite data\S2C_MSIL2A_20250330T045721_N0511_R119_T44QPE_20250330T100109.SAFE\GRANULE\L2A_T44QPE_A002949_20250330T051221\IMG_DATA\R10m\B08_10m.jp2"
B04_T1 = r"E:\Satellite data\S2B_MSIL2A_20260320T045649_N0512_R119_T44QPE_20260320T084706.SAFE\GRANULE\L2A_T44QPE_A047192_20260320T050651\IMG_DATA\R10m\B04_10m.jp2"
B08_T1 = r"E:\Satellite data\S2B_MSIL2A_20260320T045649_N0512_R119_T44QPE_20260320T084706.SAFE\GRANULE\L2A_T44QPE_A047192_20260320T050651\IMG_DATA\R10m\B08_10m.jp2"


start = time.time()

total_tiles = 0
important_tiles = 0

with rasterio.open(B04_T0) as r0, \
     rasterio.open(B08_T0) as n0, \
     rasterio.open(B04_T1) as r1, \
     rasterio.open(B08_T1) as n1:

    for row in range(0, r0.height, TILE_SIZE):
        for col in range(0, r0.width, TILE_SIZE):

            h = min(TILE_SIZE, r0.height - row)
            w = min(TILE_SIZE, r0.width - col)

            window = Window(col, row, w, h)

            
            
            red0 = r0.read(1, window=window).astype("float32") / 10000.0
            nir0 = n0.read(1, window=window).astype("float32") / 10000.0

            if np.all(red0 == 0) and np.all(nir0 == 0):
                continue

           
           
            red1 = np.zeros((h, w), dtype="float32")
            nir1 = np.zeros((h, w), dtype="float32")

           
            dst_transform = r0.window_transform(window)

            
            reproject(
                source=rasterio.band(r1, 1),
                destination=red1,
                src_transform=r1.transform,
                src_crs=r1.crs,
                dst_transform=dst_transform,
                dst_crs=r0.crs,
                resampling=Resampling.bilinear
            )

            reproject(
                source=rasterio.band(n1, 1),
                destination=nir1,
                src_transform=n1.transform,
                src_crs=n1.crs,
                dst_transform=dst_transform,
                dst_crs=r0.crs,
                resampling=Resampling.bilinear
            )

            red1 /= 10000.0
            nir1 /= 10000.0

            
            ndvi0 = (nir0 - red0) / (nir0 + red0 + 1e-6)
            ndvi1 = (nir1 - red1) / (nir1 + red1 + 1e-6)

            change = np.nan_to_num(ndvi1 - ndvi0)

            
            score = np.mean(np.abs(change) > THRESHOLD)

            if score > 0.10:
                important_tiles += 1

            total_tiles += 1

end = time.time()


print("\n--------------------------")
print(f"Total tiles: {total_tiles}")
print(f"Important tiles: {important_tiles}")
print(f"Compression ratio: {important_tiles / total_tiles:.2f}")
print(f"Time: {end - start:.2f} sec")
print(f"Speed: {total_tiles / (end - start):.2f} tiles/sec")    


--------------------------
Total tiles: 484
Important tiles: 88
Compression ratio: 0.18
Time: 11.91 sec
Speed: 40.64 tiles/sec
